# Levy + Neighbor Joint Model Plan

这个 notebook 只写实现方案，不直接改动现有拟合流程。

当前约束固定为：

- 只做 `graph step length`
- `L` 的支持集使用所有观测到的图步长可能值
- 颜色项继续使用 `canonical effective colors`
- 每个被试单独拟合自己的 `b_i` 和 `beta_i`


## 1. Model Definition

对每个被试 `i`，每一步 `t` 的 action 定义为：

- `action_it = (piece_it, color_it)`
- `state_it` 为当前棋盘状态
- `L_it` 为上一步位置到本步位置的图最短路步长

参数：

- `b_i`: 该被试的 Levy 指数
- `beta_i`: 该被试在固定步长条件下选择 piece 的 softmax 权重

单步概率分解为：

$$
p(action_{it} \mid state_{it}, b_i, \beta_i)
= p(L_{it} \mid b_i)\; p(piece_{it} \mid state_{it}, L_{it}, \beta_i)\; p(color_{it} \mid piece_{it}, state_{it})
$$

三部分含义：

1. `p(L_it | b_i)`
   在全局固定的图步长支持集上定义 Levy 型分布。

2. `p(piece_it | state_it, L_it, beta_i)`
   在当前 state 下，所有与上一步位置图距离等于 `L_it` 的候选 piece 中，按照已填色邻居个数做 softmax。

3. `p(color_it | piece_it, state_it)`
   使用 canonical effective colors。


## 2. Graph-Length Support Set

步长支持集取所有观测到的图步长可能值：

$$
\mathcal{L} = \{L : L \text{ appears in observed valid transitions}\}
$$

按现有数据，图步长目前看起来是离散整数值，例如：

`{1, 2, 3, ..., L_max}`

Levy 步长项在这个固定支持集上归一化：

$$
p(L \mid b_i) = \frac{L^{-b_i}}{\sum_{\ell \in \mathcal{L}} \ell^{-b_i}}
$$

这里 `b_i` 建议先约束在 `(1, 3)` 内直接优化，而不是一开始就上层级 Beta 分布。层级分布可以作为后续扩展。


## 3. Piece Choice Under Fixed Length

先固定观察到的步长 `L_it`，定义候选 piece 集合：

$$
\mathcal{P}(state_{it}, L_{it}) = \{p : p \text{ is uncolored and graph-distance from previous piece equals } L_{it}\}
$$

对每个候选 piece，定义效用：

$$
U(p) = \beta_i \cdot N_{colored}(p)
$$

其中 `N_colored(p)` 表示该 piece 当前已填色邻居个数。

softmax 概率：

$$
p(piece_{it}=p \mid state_{it}, L_{it}, \beta_i) = \frac{\exp(U(p))}{\sum_{q \in \mathcal{P}(state_{it}, L_{it})} \exp(U(q))}
$$

如果某一步观测到的 `L_it` 在当前 state 下没有可选 piece，需要单独记为数据异常并排查，因为理论上观测到的真实 piece 一定在候选集中。


## 4. Color Term: Canonical Effective Colors

颜色项继续沿用 `fit_softmax.py` 里的 canonical effective colors：

- 当前已使用且合法的颜色，每种单独算一个选项
- 当前未使用且合法的颜色，合并为一个“新颜色”类别

因此：

$$
p(color_{it} \mid piece_{it}, state_{it}) = \frac{1}{n_{effective}(piece_{it}, state_{it})}
$$

其中 `n_effective` 可直接复用现有 `count_effective_colors(...)` 的逻辑。


## 5. Per-Participant Objective

对单个被试 `i`，总 log-likelihood 为：

$$
\mathcal{L}_i(b_i, \beta_i)
= \sum_t \log p(L_{it} \mid b_i)
+ \sum_t \log p(piece_{it} \mid state_{it}, L_{it}, \beta_i)
+ \sum_t \log p(color_{it} \mid piece_{it}, state_{it})
$$

要最大化的是这个被试自己的 log-likelihood。

等价地最小化 `negative log-likelihood`。


## 6. Data Structure Needed

建议为每个有效 step 构建一条记录，字段至少包括：

- `participant`
- `round_label`
- `step_in_round`
- `prev_region`
- `chosen_region`
- `chosen_color`
- `graph_length`：本步真实图步长
- `candidate_regions_at_same_length`：所有与真实步长相同的候选 piece
- `colored_neighbor_count_by_candidate`
- `num_effective_colors_for_chosen_piece`
- `current_colors`
- `adjacency`

其中：

- `graph_length` 已经可以从当前 `step_length_models.py` 的预计算距离矩阵得到
- 颜色项需要复用 `fit_softmax.py` 中的 `count_effective_colors(...)`


## 7. Suggested Code Layout

建议新增一个独立文件，例如：

- `model code/fitting/fit_levy_neighbor_joint.py`

里面包含这些函数：

1. `build_joint_steps(...)`
   回放数据，构建联合模型需要的 step-level records。

2. `global_graph_length_support(joint_steps)`
   从所有观测 step 中抽取全局图步长支持集。

3. `log_p_length(L, b, support)`
   在固定支持集上计算 Levy 长度项。

4. `log_p_piece(step, beta)`
   在固定步长条件下，对候选 piece 的 softmax 计算 log-prob。

5. `log_p_color(step)`
   使用 canonical effective colors 计算颜色项。

6. `neg_log_likelihood_joint(params, participant_steps, support)`
   参数 `params = [b, beta]`。

7. `fit_participant_joint(filepath, maps, support)`
   对单个被试拟合 `(b_i, beta_i)`。

8. `fit_all_participants_joint(...)`
   批量拟合所有被试并输出表格。


## 8. Optimization Strategy

建议先用最简单稳定的方式：

- 参数：`b_i` 和 `beta_i`
- 约束：`1 < b_i < 3`
- `beta_i` 可先不加硬约束
- 优化器可先用 `scipy.optimize.minimize` with `L-BFGS-B` 或 `Nelder-Mead`

推荐起点：

- `b_i = 1.5`
- `beta_i = 1.0`

推荐输出：

- `b_hat`
- `beta_hat`
- `ll`
- `nll`
- `aic`
- `bic`
- 各部分 log-likelihood 拆分：
  - `ll_length`
  - `ll_piece`
  - `ll_color`


## 9. Diagnostics to Add

为了确认模型工作正常，建议同时输出：

- 每个被试的 `b_hat` 分布
- 每个被试的 `beta_hat` 分布
- `ll_length / ll_piece / ll_color` 的相对贡献
- 逐步 `log p(L|b)` 分布
- 逐步 `log p(piece|state,L,beta)` 分布
- 逐步总 `loglik` 分布

还可以和现有模型做比较：

- 只有长度项的 Levy 模型
- 只有 piece softmax 的邻居模型
- 当前这个联合模型


## 10. Practical Notes Before Coding

实现时最容易出错的地方：

1. `L` 的定义要严格使用图最短路，不是欧氏距离。

2. piece 的 softmax 候选集必须是“未着色且图步长等于真实 `L`”的所有 piece。

3. 颜色项必须和现有 `fit_softmax.py` 保持一致，否则联合模型与旧模型不可直接比较。

4. 全局步长支持集固定后，不要逐 step 重新归一化 `p(L|b)`。

5. 如果某一步在当前 state 下找不到同步长候选集，说明数据回放或步长计算有 bug，应先修数据构建逻辑。


In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'fit_softmax.py').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'model code' / 'fitting'
sys.path.insert(0, str(NOTEBOOK_DIR))

from fit_softmax import count_effective_colors, build_all_maps, parse_csv
from step_length_models import build_transition_table

print('Plan notebook loaded. No fitting is executed here.')
